# nb00 — Poisoned Model Reference

Reproduces the poisoned model's predictions on the 2 000 test images and verifies they
match `sample_submission.csv` (which IS the poisoned model's predictions).

**Purpose:** confirm that our `build_cfg` / `load_for_inference` / inference loop is
bit-for-bit equivalent to the host's pipeline before we change anything.

**No LB submit** — this would just reproduce the poisoned-model score that is already on the board.

## 1. Install Detectron2

In [ ]:
!pip install -q 'setuptools<81'
!pip install -q 'git+https://github.com/facebookresearch/detectron2.git'

## 2. Imports

In [ ]:
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
from detectron2 import model_zoo
from detectron2.config import get_cfg
from detectron2.engine import DefaultPredictor
from tqdm import tqdm

## 3. Configuration

In [ ]:
# ── Competition data paths (handles both Kaggle mount styles) ──
def _find_comp_data():
    competitions = Path("/kaggle/input/competitions/neural-debris-removal-in-streak-detection-models")
    standard     = Path("/kaggle/input/neural-debris-removal-in-streak-detection-models")
    return competitions if competitions.exists() else standard

COMP_ROOT         = _find_comp_data()
POISONED_WEIGHTS  = str(COMP_ROOT / "poisoned_model/poisoned_model.pth")
TEST_DIR          = str(COMP_ROOT / "test_set/test_set")
SAMPLE_SUBMISSION = str(COMP_ROOT / "sample_submission.csv")
SUBMISSION_PATH   = "/kaggle/working/submission.csv"

# ── Model architecture (MUST match poisoned model training config) ──
BASE_CONFIG          = "COCO-Detection/retinanet_R_50_FPN_3x.yaml"
ANCHOR_ASPECT_RATIOS = [0.1, 0.2, 0.5, 1.0, 2.0, 5.0, 10.0]
ANCHOR_SIZES         = [[16], [32], [64], [128], [256]]
NUM_CLASSES          = 1
CONF_THRESH          = 0.2
IMG_W = IMG_H        = 1024

print(f"Competition data root: {COMP_ROOT}")
print(f"Poisoned weights:      {POISONED_WEIGHTS}")
print(f"Test dir:              {TEST_DIR}")

## 4. Core functions (mirrored from depoison_core.py)

In [ ]:
def build_cfg(weights_path, score_thresh=0.2):
    """Return a Detectron2 CfgNode matching the poisoned model's architecture exactly."""
    cfg = get_cfg()
    cfg.merge_from_file(model_zoo.get_config_file(BASE_CONFIG))
    cfg.MODEL.WEIGHTS                        = str(weights_path)
    cfg.MODEL.RETINANET.NUM_CLASSES          = NUM_CLASSES
    cfg.MODEL.RETINANET.SCORE_THRESH_TEST    = score_thresh
    cfg.MODEL.ANCHOR_GENERATOR.ASPECT_RATIOS = [ANCHOR_ASPECT_RATIOS]
    cfg.MODEL.ANCHOR_GENERATOR.SIZES         = ANCHOR_SIZES
    return cfg


def load_for_inference(path):
    """Load a 16-bit PNG and return float32 HWC array in [0, 255] with 3 channels."""
    im = cv2.imread(str(path), cv2.IMREAD_UNCHANGED)
    if im.dtype == np.uint16:
        im = im.astype(np.float32) / 65535.0
    im = np.clip(im * 255, 0, 255).astype(np.float32)
    if im.ndim == 2:
        im = np.repeat(im[:, :, None], 3, axis=2)
    return im

## 5. Build predictor and run inference

In [ ]:
cfg = build_cfg(POISONED_WEIGHTS, score_thresh=CONF_THRESH)
predictor = DefaultPredictor(cfg)

test_files = sorted(Path(TEST_DIR).glob("*.png"))
print(f"Running inference on {len(test_files)} images...")

rows = []
for img_path in tqdm(test_files, desc="Inference"):
    im  = load_for_inference(img_path)
    out = predictor(im)["instances"].to("cpu")
    boxes  = out.pred_boxes.tensor.numpy()
    scores = out.scores.numpy()

    parts = []
    for (x1, y1, x2, y2), s in zip(boxes, scores):
        x1 = float(np.clip(x1, 0, IMG_W))
        y1 = float(np.clip(y1, 0, IMG_H))
        x2 = float(np.clip(x2, 0, IMG_W))
        y2 = float(np.clip(y2, 0, IMG_H))
        w, h = max(0.0, x2 - x1), max(0.0, y2 - y1)
        if w == 0 or h == 0:
            continue
        parts.extend([
            f"{float(s):.6f}", f"{x1:.2f}", f"{y1:.2f}", f"{w:.2f}", f"{h:.2f}"
        ])

    rows.append({"image_id": img_path.stem, "prediction_string": " ".join(parts) or " "})

submission = pd.DataFrame(rows)
submission.insert(0, "id", range(len(submission)))
submission.to_csv(SUBMISSION_PATH, index=False)
print(f"Wrote {SUBMISSION_PATH}  ({len(submission)} rows)")
submission.head()

## 6. Equivalence check vs sample_submission.csv

If our pipeline reproduces the poisoned model exactly, all `prediction_string` values should
match (within float rounding).  Any structural mismatch (different detection count per image)
means the architecture config or weights path is wrong.

In [ ]:
sample = pd.read_csv(SAMPLE_SUBMISSION)
ours   = pd.read_csv(SUBMISSION_PATH)

print("=== Equivalence Check ===")
print(f"sample_submission rows : {len(sample)}")
print(f"our submission rows    : {len(ours)}")

assert len(sample) == 2000 and len(ours) == 2000, "Row count mismatch!"

# Merge on image_id (handles potential ordering differences)
merged = sample.merge(ours, on="image_id", suffixes=("_sample", "_ours"))
assert len(merged) == 2000, f"Merge lost rows: {len(merged)}"

# Normalise: sample_submission uses "" for no-det rows, we use " "
def normalise(s):
    return "" if (not isinstance(s, str) or s.strip() == "") else s.strip()

merged["pred_sample"] = merged["prediction_string_sample"].apply(normalise)
merged["pred_ours"]   = merged["prediction_string_ours"].apply(normalise)

exact = (merged["pred_sample"] == merged["pred_ours"]).sum()
mismatches = merged[merged["pred_sample"] != merged["pred_ours"]]

print(f"\nExact string matches : {exact} / 2000")
print(f"Mismatched rows      : {len(mismatches)}")

# Count detections per image to check structural match
def count_dets(s):
    s = normalise(s)
    return len(s.split()) // 5 if s else 0

merged["n_sample"] = merged["prediction_string_sample"].apply(count_dets)
merged["n_ours"]   = merged["prediction_string_ours"].apply(count_dets)
det_diff = (merged["n_sample"] != merged["n_ours"]).sum()

print(f"Images with different detection count : {det_diff}")
print(f"\nTotal detections — sample: {merged['n_sample'].sum()}, ours: {merged['n_ours'].sum()}")

if len(mismatches) > 0:
    print("\nFirst 5 mismatches:")
    for _, row in mismatches.head(5).iterrows():
        print(f"  image_id={row['image_id']}")
        print(f"    sample : {str(row['prediction_string_sample'])[:120]}")
        print(f"    ours   : {str(row['prediction_string_ours'])[:120]}")

if det_diff == 0:
    print("\n✓ Detection structure matches. Pipeline reproduces the poisoned model.")
    if len(mismatches) == 0:
        print("✓ Exact string match confirmed.")
    else:
        print("⚠ Minor float-rounding differences only (same detection count).")
else:
    print("\n✗ STRUCTURAL MISMATCH — check architecture config or weights path!")